# 04 - Prototype webpage (Gradio)

Takes a latitude/longitude + start month, pulls real Sentinel-1 data for 12 months, runs your trained model, and shows the results as an interactive webpage with a shareable link. Also includes a Case Study tab comparing your model against Google Dynamic World (the independent source).

**Before running:** make sure you've run Earth Engine authentication at least once in this Colab session (same as your data collection notebooks). If your Earth Engine project ID isn't `lulc-sentinel1`, change it in Cell 1.

**Cell 1:** Install libraries, mount Drive, authenticate Earth Engine.

In [ ]:
!pip install -q gradio earthengine-api rasterio segmentation-models-pytorch torchgeo plotly

import ee
import os
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import requests
import rasterio
import gradio as gr

from google.colab import drive
drive.mount('/content/drive')

ee.Authenticate()
ee.Initialize(project='lulc-sentinel1')  # change this if your Earth Engine project id is different

DRIVE_FOLDER = '/content/drive/MyDrive/lulc_dataset'
CKPT_DIR = f'{DRIVE_FOLDER}/All Model Weights'
print('Earth Engine ready. Checkpoint folder:', CKPT_DIR)


**Cell 2:** Classes, colors for the maps, and loading your trained model. `MODEL_RUN_NAME` defaults to your original UnetPlusPlus checkpoint. If `03f` produces a better variant you want to use instead, just change this one line to that run_name.

In [ ]:
NUM_CLASSES = 9
CLASS_NAMES = [
    "Tree cover", "Shrubland", "Grassland", "Cropland", "Built-up",
    "Bare/sparse vegetation", "Permanent water", "Herbaceous wetland", "Mangroves",
]
CLASS_COLORS = {
    "Tree cover": (0, 100, 0), "Shrubland": (154, 205, 50), "Grassland": (173, 255, 47),
    "Cropland": (240, 230, 140), "Built-up": (220, 20, 60), "Bare/sparse vegetation": (210, 180, 140),
    "Permanent water": (30, 60, 200), "Herbaceous wetland": (0, 191, 191), "Mangroves": (102, 51, 0),
}
COLOR_LUT = np.array([CLASS_COLORS[name] for name in CLASS_NAMES], dtype=np.uint8)

import segmentation_models_pytorch as smp
import torchgeo.models as tgm

def _load_resnet50_sar_encoder(smp_model, weights_name, freeze=False):
    weights_enum = getattr(tgm.ResNet50_Weights, weights_name)
    state_dict = weights_enum.get_state_dict(progress=True)
    smp_model.encoder.load_state_dict(state_dict, strict=True)
    if freeze:
        for p in smp_model.encoder.parameters():
            p.requires_grad = False
    return smp_model

def build_model():
    model = smp.UnetPlusPlus(encoder_name='resnet50', encoder_weights=None, in_channels=2, classes=NUM_CLASSES)
    return _load_resnet50_sar_encoder(model, 'SENTINEL1_ALL_MOCO', freeze=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_RUN_NAME = 'unetplusplus_ssl4eo_moco'  # swap to a 03f variant's run_name if that one wins

model = build_model()
model.load_state_dict(torch.load(f'{CKPT_DIR}/{MODEL_RUN_NAME}_best.pt', map_location=device))
model = model.to(device).eval()
print(f'Loaded model: {MODEL_RUN_NAME}, device: {device}')


**Cell 3:** Pull real Sentinel-1 data for a given lat/lon + month from Earth Engine, build the monthly composite, and apply the exact same preprocessing as training (dB clip -25 to 5, normalize 0-1). Uses `getDownloadURL`, an interactive request, not the slow batch export you used for the original dataset.

In [ ]:
PATCH_SIZE_PX = 256
SCALE_M = 10
PATCH_SIZE_M = PATCH_SIZE_PX * SCALE_M


def get_sentinel1_composite(lat, lon, year, month):
    point = ee.Geometry.Point([lon, lat])
    aoi = point.buffer(PATCH_SIZE_M / 2).bounds()

    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, 'month')

    collection = (ee.ImageCollection('COPERNICUS/S1_GRD')
                  .filterBounds(aoi)
                  .filterDate(start, end)
                  .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
                  .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
                  .filter(ee.Filter.eq('instrumentMode', 'IW'))
                  .select(['VV', 'VH']))

    n_images = collection.size().getInfo()
    if n_images == 0:
        raise ValueError(f'No Sentinel-1 images found for {year}-{month:02d} at this location.')

    composite = collection.median().clip(aoi)
    url = composite.getDownloadURL({'region': aoi, 'scale': SCALE_M, 'format': 'GEO_TIFF'})
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()

    with rasterio.io.MemoryFile(resp.content) as memfile:
        with memfile.open() as dataset:
            arr = dataset.read().astype(np.float32)

    return arr, n_images


def preprocess_composite(arr):
    arr = np.clip(arr, -25, 5)
    arr = (arr - (-25)) / (5 - (-25))

    _, h, w = arr.shape
    if (h, w) != (PATCH_SIZE_PX, PATCH_SIZE_PX):
        out = np.zeros((2, PATCH_SIZE_PX, PATCH_SIZE_PX), dtype=np.float32)
        ch, cw = min(h, PATCH_SIZE_PX), min(w, PATCH_SIZE_PX)
        out[:, :ch, :cw] = arr[:, :ch, :cw]
        arr = out
    return arr

print('Earth Engine retrieval functions ready.')


**Cell 4:** Run the model on a composite, and turn its prediction into class percentages and a colored image.

In [ ]:
@torch.no_grad()
def predict_lulc(arr_2ch):
    x = torch.from_numpy(arr_2ch).float().unsqueeze(0).to(device)
    logits = model(x)
    return logits.argmax(dim=1).squeeze(0).cpu().numpy()


def class_percentages(pred):
    total = pred.size
    return {CLASS_NAMES[i]: float((pred == i).sum()) / total * 100 for i in range(NUM_CLASSES)}


def colorize(pred):
    return COLOR_LUT[pred]

print('Inference + percentage functions ready.')


**Cell 5:** The 12-month pipeline: fetch, preprocess, predict, repeat for 12 months starting from the given month, reporting progress along the way (this is what drives the live status messages on the webpage). Also the month-to-month and total change calculation.

In [ ]:
def shift_month(year, month, offset):
    total = (year * 12 + (month - 1)) + offset
    return total // 12, total % 12 + 1


def run_pipeline(lat, lon, start_year, start_month, progress=None):
    monthly_colored = []
    monthly_pct = []
    labels = []

    for i in range(12):
        y, m = shift_month(start_year, start_month, i)
        label = f'{y}-{m:02d}'
        labels.append(label)
        if progress is not None:
            progress(i / 12, desc=f'Fetching Sentinel-1 for {label} ({i+1}/12)...')

        arr, n_images = get_sentinel1_composite(lat, lon, y, m)
        arr = preprocess_composite(arr)

        if progress is not None:
            progress((i + 0.5) / 12, desc=f'Running model on {label} ({i+1}/12)...')

        pred = predict_lulc(arr)
        monthly_colored.append(colorize(pred))
        monthly_pct.append(class_percentages(pred))

    if progress is not None:
        progress(1.0, desc='Done.')

    pct_df = pd.DataFrame(monthly_pct, index=labels)
    return monthly_colored, pct_df


def compute_changes(pct_df):
    # Month 1 has no earlier month to compare against, so instead of dropping it
    # (which made the table start at month 2 and look like month 1 was missing),
    # it's kept as a row of 0.0 everywhere: the correct, honest value for "change
    # from itself". That keeps all 12 months in the table, in order.
    month_to_month = pct_df.diff().fillna(0.0)
    total_change = pct_df.iloc[-1] - pct_df.iloc[0]
    return month_to_month, total_change

print('Pipeline ready.')


**Cell 6:** Dynamic World retrieval and class mapping, for the Case Study tab. Dynamic World's 9 classes are mapped to your 9 training classes (its 'snow_and_ice' class has no match in your scheme, those pixels are excluded from the agreement score, not counted as wrong).

In [ ]:
DW_TO_OURS = {
    0: 'Permanent water', 1: 'Tree cover', 2: 'Grassland', 3: 'Herbaceous wetland',
    4: 'Cropland', 5: 'Shrubland', 6: 'Built-up', 7: 'Bare/sparse vegetation', 8: None,
}


def get_dynamic_world(lat, lon, year, month):
    point = ee.Geometry.Point([lon, lat])
    aoi = point.buffer(PATCH_SIZE_M / 2).bounds()
    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, 'month')

    collection = (ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
                  .filterBounds(aoi).filterDate(start, end).select('label'))
    if collection.size().getInfo() == 0:
        return None

    dw_mode = collection.mode().clip(aoi)
    url = dw_mode.getDownloadURL({'region': aoi, 'scale': SCALE_M, 'format': 'GEO_TIFF'})
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    with rasterio.io.MemoryFile(resp.content) as memfile:
        with memfile.open() as dataset:
            dw_arr = dataset.read(1)

    if dw_arr.shape != (PATCH_SIZE_PX, PATCH_SIZE_PX):
        out = np.zeros((PATCH_SIZE_PX, PATCH_SIZE_PX), dtype=dw_arr.dtype)
        h, w = dw_arr.shape
        ch, cw = min(h, PATCH_SIZE_PX), min(w, PATCH_SIZE_PX)
        out[:ch, :cw] = dw_arr[:ch, :cw]
        dw_arr = out
    return dw_arr


def dw_to_our_classes(dw_arr):
    out = np.full(dw_arr.shape, -1, dtype=np.int64)
    for dw_class, our_name in DW_TO_OURS.items():
        if our_name is None:
            continue
        out[dw_arr == dw_class] = CLASS_NAMES.index(our_name)
    return out


def run_case_study(lat, lon, year, month):
    arr, _ = get_sentinel1_composite(lat, lon, year, month)
    arr = preprocess_composite(arr)
    our_pred = predict_lulc(arr)

    dw_arr = get_dynamic_world(lat, lon, year, month)
    if dw_arr is None:
        return colorize(our_pred), None, None, None

    dw_mapped = dw_to_our_classes(dw_arr)
    valid = dw_mapped != -1
    agreement = float((our_pred[valid] == dw_mapped[valid]).sum()) / max(1, valid.sum()) * 100

    # Per-class breakdown: of the pixels Dynamic World itself labeled as class X, what %
    # did our model also call class X? A class Dynamic World never found here is skipped,
    # there's nothing to compare it against.
    per_class_agreement = {}
    for i, name in enumerate(CLASS_NAMES):
        class_mask = valid & (dw_mapped == i)
        n = int(class_mask.sum())
        if n > 0:
            match_pct = float((our_pred[class_mask] == i).sum()) / n * 100
            per_class_agreement[name] = (match_pct, n)

    dw_colored = np.zeros((*dw_arr.shape, 3), dtype=np.uint8)
    dw_colored[valid] = COLOR_LUT[dw_mapped[valid]]

    return colorize(our_pred), dw_colored, agreement, per_class_agreement

print('Dynamic World comparison ready.')


**Cell 7:** Chart helpers. The legend is now clean HTML swatches (no more overlapping rotated
text), and the layout helpers build: a plain-language summary of the result, a composition bar
chart for the most recent month, a stacked area chart for the full 12-month trend (classes that
are basically absent are automatically hidden), and a green/red bar chart of the total change.
These are what the "Final Result" section of the page actually shows, arranged side by side rather
than stacked one below another.

In [ ]:
import plotly.graph_objects as go


def make_legend_html():
    """Clean color key as HTML swatches: a colored square + label per class, wrapped in a
    rounded panel that matches the page theme. Replaces the old matplotlib legend, whose
    rotated, overlapping text labels looked broken."""
    items = []
    for name in CLASS_NAMES:
        r, g, b = CLASS_COLORS[name]
        items.append(
            '<div style="display:flex;align-items:center;margin:5px 14px 5px 0;">'
            f'<span style="display:inline-block;width:16px;height:16px;border-radius:4px;'
            f'background:rgb({r},{g},{b});margin-right:7px;border:1px solid rgba(255,255,255,0.35);"></span>'
            f'<span style="color:#F5F5F5;font-size:13px;">{name}</span></div>'
        )
    return (
        '<div style="display:flex;flex-wrap:wrap;align-items:center;'
        'background:#19376D;padding:12px 16px;border-radius:10px;">'
        + ''.join(items) + '</div>'
    )


def make_architecture_diagram_html():
    """A boxes-and-arrows pipeline diagram for the 'How This Works' tab. Every box is the
    exact same fixed size (icon + short title + short subtitle) so nothing wraps badly or
    looks inconsistent, and the raw model run_name (long and code-like) is kept out of the
    diagram itself, just an icon + plain label, with the real name given in the card below."""
    steps = [
        ('📍', 'Your Input', 'lat / lon + month'),
        ('🛰️', 'Earth Engine', 'real Sentinel-1 VV+VH'),
        ('🧮', 'Composite', 'median, like training'),
        ('🧠', 'AI Model', 'UnetPlusPlus, SAR-trained'),
        ('🗺️', 'Prediction', f'{NUM_CLASSES} classes / pixel'),
        ('📊', 'Result', 'area % + change'),
    ]
    box_style = (
        'background:#19376D;border:2px solid #FFD700;border-radius:12px;'
        'width:128px;height:108px;flex:0 0 128px;'
        'display:flex;flex-direction:column;align-items:center;justify-content:center;'
        'text-align:center;box-sizing:border-box;padding:6px;'
    )
    parts = []
    for i, (icon, title, sub) in enumerate(steps):
        parts.append(
            f'<div style="{box_style}">'
            f'<div style="font-size:22px;line-height:1;">{icon}</div>'
            f'<div style="color:#FFD700;font-weight:700;font-size:14px;margin-top:4px;">{title}</div>'
            f'<div style="color:#F5F5F5;font-size:11px;margin-top:2px;line-height:1.3;">{sub}</div>'
            '</div>'
        )
        if i < len(steps) - 1:
            parts.append(
                '<div style="color:#FFD700;font-size:24px;flex:0 0 auto;padding:0 2px;">&#8594;</div>'
            )
    return (
        '<div style="display:flex;align-items:center;flex-wrap:wrap;justify-content:center;'
        'gap:4px;padding:20px;background:#0B2447;border:1px solid #19376D;border-radius:14px;">'
        + ''.join(parts) + '</div>'
    )


def make_info_card_html(icon, title, lines, formula_lines=None):
    """A styled 'How This Works' card: icon + title header with a gold divider, each line
    as a bulleted row (bold spans via <strong>, since this is raw HTML not Markdown), and
    optional formula_lines rendered as monospace highlighted chips so they read as formulas,
    not as plain sentences. Replaces the old plain gr.Markdown bullet list."""
    rows = ''.join(
        f'<div style="display:flex;gap:8px;align-items:flex-start;margin-bottom:7px;">'
        f'<span style="color:#FFD700;font-size:13px;line-height:1.5;">▸</span>'
        f'<span style="color:#F5F5F5;font-size:13.5px;line-height:1.5;">{line}</span></div>'
        for line in lines
    )
    formula_html = ''
    if formula_lines:
        chips = ''.join(
            f'<div style="background:#0B2447;border:1px solid #FFD700;border-radius:6px;'
            f'padding:6px 10px;margin-bottom:6px;font-family:monospace;font-size:12.5px;'
            f'color:#FFD700;">{f}</div>'
            for f in formula_lines
        )
        formula_html = f'<div style="margin-top:8px;">{chips}</div>'
    return (
        '<div style="background:#19376D;border:1px solid rgba(255,215,0,0.4);'
        'border-radius:12px;padding:16px 18px;height:100%;box-sizing:border-box;">'
        '<div style="display:flex;align-items:center;gap:8px;margin-bottom:10px;">'
        f'<span style="font-size:20px;">{icon}</span>'
        f'<span style="color:#FFD700;font-weight:700;font-size:15px;">{title}</span>'
        '</div>'
        '<div style="border-top:1px solid rgba(255,215,0,0.3);margin-bottom:10px;"></div>'
        f'{rows}{formula_html}'
        '</div>'
    )


def make_limitations_html(lines):
    """A styled warning-style panel for the limitations list: amber border and a warning
    icon, instead of a plain flat dark bar with bare bullet text."""
    rows = ''.join(
        f'<div style="display:flex;gap:10px;align-items:flex-start;margin-bottom:9px;">'
        f'<span style="color:#FFA500;font-size:14px;flex:0 0 auto;">⚠️</span>'
        f'<span style="color:#F5F5F5;font-size:14px;line-height:1.5;">{line}</span></div>'
        for line in lines
    )
    return (
        '<div style="background:#2A1F0F;border:2px solid #FFA500;border-radius:10px;'
        'padding:16px 20px;">'
        '<div style="color:#FFA500;font-weight:700;font-size:15px;margin-bottom:10px;">'
        '⚠️ Known Limitations</div>'
        f'{rows}'
        '</div>'
    )


def make_agreement_breakdown_html(per_class_agreement, min_pixels=50, max_rows=6):
    """Compact per-class agreement pills for the Case Study tab. Only classes Dynamic World
    actually labeled a meaningful amount of (>= min_pixels) are shown, capped at max_rows and
    sorted by how much area each one covers, so this highlights the classes that actually
    matter for this one scene instead of a long list padded out with one-pixel noise classes."""
    rows = sorted(per_class_agreement.items(), key=lambda kv: -kv[1][1])
    rows = [r for r in rows if r[1][1] >= min_pixels][:max_rows]
    if not rows:
        return ''
    chips = []
    for name, (pct, _n) in rows:
        r, g, b = CLASS_COLORS[name]
        pct_color = '#3CB371' if pct >= 70 else ('#FFD700' if pct >= 40 else '#E74C3C')
        chips.append(
            '<div style="display:flex;align-items:center;gap:8px;background:#0B2447;'
            'border:1px solid rgba(255,255,255,0.15);border-radius:8px;padding:7px 12px;">'
            f'<span style="display:inline-block;width:12px;height:12px;border-radius:3px;'
            f'background:rgb({r},{g},{b});flex:0 0 auto;"></span>'
            f'<span style="color:#F5F5F5;font-size:13px;">{name}</span>'
            f'<span style="color:{pct_color};font-weight:700;font-size:13px;margin-left:auto;'
            f'padding-left:10px;">{pct:.0f}%</span></div>'
        )
    return (
        '<div style="margin-top:10px;">'
        '<div style="color:#FFD700;font-size:12.5px;font-weight:600;margin-bottom:6px;">'
        'Agreement by class (classes Dynamic World found a meaningful amount of here)</div>'
        '<div style="display:flex;flex-wrap:wrap;gap:8px;">' + ''.join(chips) + '</div>'
        '</div>'
    )


def generate_summary(pct_df, total_change):
    """Plain-language answer: what this place mainly is, what changed the most, and how
    stable it looks overall. This is meant to be read first, before any chart."""
    last_month = pct_df.iloc[-1]
    dominant = last_month[last_month > 1.0].sort_values(ascending=False)
    top_classes = dominant.index.tolist()[:3]
    if top_classes:
        top_str = ', '.join(f'{name} ({last_month[name]:.0f}%)' for name in top_classes)
    else:
        top_str = 'no single class clearly dominant'

    meaningful_change = total_change[total_change.abs() > 0.2]
    if len(meaningful_change) == 0:
        change_str = 'no class changed by more than 0.2 percentage points.'
    else:
        biggest_name = meaningful_change.abs().sort_values(ascending=False).index[0]
        biggest_val = total_change[biggest_name]
        direction = 'increased' if biggest_val > 0 else 'decreased'
        change_str = f'**{biggest_name}** {direction} the most, by {abs(biggest_val):.1f} percentage points.'

    overall_abs_change = total_change.abs().sum()
    if overall_abs_change < 2:
        stability = 'Overall, this area looks largely **stable** across the 12 months.'
    elif overall_abs_change < 8:
        stability = 'Overall, this area shows **moderate** change across the 12 months.'
    else:
        stability = 'Overall, this area shows **significant** change across the 12 months.'

    return (
        f"### What this result shows\n"
        f"- **This location is mainly:** {top_str}\n"
        f"- **Biggest change (month 1 to month 12):** {change_str}\n"
        f"- {stability}"
    )


PANEL_FACECOLOR = '#19376D'
PANEL_TEXTCOLOR = '#F5F5F5'


def _style_panel(fig, ax):
    """Make a matplotlib figure sit visually inside the dark-blue themed page instead of
    showing up as a bright white rectangle."""
    fig.patch.set_facecolor(PANEL_FACECOLOR)
    ax.set_facecolor(PANEL_FACECOLOR)
    ax.tick_params(colors=PANEL_TEXTCOLOR, labelsize=9)
    ax.xaxis.label.set_color(PANEL_TEXTCOLOR)
    ax.yaxis.label.set_color(PANEL_TEXTCOLOR)
    ax.title.set_color(PANEL_TEXTCOLOR)
    for spine in ax.spines.values():
        spine.set_color(PANEL_TEXTCOLOR)
        spine.set_alpha(0.3)


def make_composition_chart(pct_df):
    """Horizontal bar: how the area is made up in the LAST month, classes under 0.3% hidden
    so the chart isn't full of flat zero bars. Sized to sit side by side with the change chart."""
    row = pct_df.iloc[-1]
    row = row[row > 0.3].sort_values(ascending=True)
    colors = [tuple(c / 255 for c in CLASS_COLORS[name]) for name in row.index]
    fig, ax = plt.subplots(figsize=(6, max(2.2, 0.55 * len(row))))
    bars = ax.barh(row.index, row.values, color=colors)
    for bar, val in zip(bars, row.values):
        ax.text(val + 0.5, bar.get_y() + bar.get_height() / 2, f'{val:.1f}%',
                va='center', fontsize=9, color=PANEL_TEXTCOLOR)
    ax.set_xlabel('Area %')
    ax.set_title(f'Composition in {pct_df.index[-1]} (latest month)')
    ax.set_xlim(0, max(list(row.values) + [1]) * 1.3)
    _style_panel(fig, ax)
    plt.tight_layout()
    return fig


def make_stacked_area_chart(pct_df):
    """Stacked area, now an interactive Plotly figure instead of a static matplotlib image,
    so hovering anywhere on a colored band pops up that class's name and its exact area %
    for that month (and every other class at that month too, via the unified hover line).
    Classes that never rise above 0.3% are dropped from the chart (not the underlying data)
    so the shape of the colored bands tells the story instead of 9 overlapping lines."""
    present_classes = [c for c in CLASS_NAMES if pct_df[c].max() > 0.3]
    if not present_classes:
        present_classes = CLASS_NAMES

    fig = go.Figure()
    for name in present_classes:
        r, g, b = CLASS_COLORS[name]
        fig.add_trace(go.Scatter(
            x=pct_df.index, y=pct_df[name],
            name=name, mode='lines', stackgroup='area',
            line=dict(width=0.5, color=f'rgb({r},{g},{b})'),
            fillcolor=f'rgb({r},{g},{b})',
            hovertemplate='%{fullData.name}: %{y:.1f}%<extra></extra>',
        ))

    fig.update_layout(
        title='Composition over 12 months (hover for exact values)',
        xaxis_title='Month', yaxis_title='Area %',
        yaxis=dict(range=[0, 100]),
        hovermode='x unified',
        paper_bgcolor=PANEL_FACECOLOR, plot_bgcolor=PANEL_FACECOLOR,
        font=dict(color=PANEL_TEXTCOLOR),
        legend=dict(bgcolor=PANEL_FACECOLOR, bordercolor='rgba(255,255,255,0.2)', borderwidth=1,
                    traceorder='reversed'),  # matches the table's left-to-right class order
        margin=dict(l=60, r=30, t=60, b=70),
        height=430,
    )
    fig.update_xaxes(tickangle=45, gridcolor='rgba(255,255,255,0.08)', zeroline=False)
    fig.update_yaxes(gridcolor='rgba(255,255,255,0.08)', zeroline=False)
    return fig


def make_total_change_chart(total_change):
    """Horizontal bar, green = grew, red = shrank. Only classes with more than 0.2 percentage
    points of total change are shown. Sized to sit side by side with the composition chart.
    The value label for each bar sits just past its OUTER tip (always further from zero,
    never toward the axis), and the x-axis gets generous padding on both sides, so long
    labels like '-2.6%' never collide with the y-axis category names anymore."""
    tc = total_change[total_change.abs() > 0.2].sort_values()
    n = len(tc)
    fig, ax = plt.subplots(figsize=(6.5, max(2.4, 0.55 * n) if n else 2.4))
    if n == 0:
        ax.text(0.5, 0.5, 'No class changed by more than 0.2 pts.',
                ha='center', va='center', fontsize=10, color=PANEL_TEXTCOLOR)
        ax.axis('off')
        fig.patch.set_facecolor(PANEL_FACECOLOR)
        return fig
    colors = ['#3CB371' if v > 0 else '#E74C3C' for v in tc.values]
    bars = ax.barh(tc.index, tc.values, color=colors)

    max_abs = max(abs(tc.values.min()), abs(tc.values.max()), 0.5)
    pad = max_abs * 0.35  # room for the text label past the bar tip
    ax.set_xlim(-(max_abs + pad), max_abs + pad)

    offset = max_abs * 0.06
    for bar, val in zip(bars, tc.values):
        x = val + offset if val >= 0 else val - offset
        ax.text(x, bar.get_y() + bar.get_height() / 2, f'{val:+.1f}%',
                va='center', ha='left' if val >= 0 else 'right',
                fontsize=9, color=PANEL_TEXTCOLOR, clip_on=False)
    ax.axvline(0, color=PANEL_TEXTCOLOR, linewidth=0.8, alpha=0.6)
    ax.set_xlabel('Change in area % (month 1 to month 12)')
    ax.set_title('Total change (green = grew, red = shrank)')
    _style_panel(fig, ax)
    plt.tight_layout()
    return fig

print('Chart helpers ready.')


**Cell 8:** The Gradio webpage itself. Three tabs: **"How This Works"** (a pipeline diagram plus
three crisp cards covering architecture, data retrieval/compositing, and the % / change formulas),
**"Run Analysis"** (a radio-button location picker with Rostock / Mumbai / Berlin / Custom, each
showing its own one-line explanation the moment you select it, so there's no separate "use preset"
click; maps beside the color key; a "Final Result" section with the composition/total-change charts
side by side and the 12-month trend full-width below; the detailed month-to-month table tucked into
a collapsed panel, now showing all 12 months in order, including month 1 as a 0.0 baseline row
instead of being dropped), and **"Case Study vs Dynamic World"**. (The Limitations section was
moved out of this page into the written report.) Running this cell launches the page and prints a
public link.

In [ ]:
PRESETS = {
    'Rostock (same-region test city)': {
        'lat': 54.0887, 'lon': 12.1414, 'year': 2023, 'month': 6,
        'desc': (
            "Rostock was never used in training, but it sits in the SAME region (Germany / "
            "Northern Europe) as the cities the model learned from. On the real held-out "
            "test set, this group scores mIoU 0.45, the easiest generalization case: a "
            "brand-new place that still looks similar to what the model already knows."
        ),
    },
    'Mumbai (dissimilar test city)': {
        'lat': 19.0760, 'lon': 72.8777, 'year': 2023, 'month': 6,
        'desc': (
            "Mumbai was never used in training and sits in a DISSIMILAR region: different "
            "climate, vegetation, and urban layout than any training city. On the real "
            "held-out test set, this group scores mIoU 0.23, less than half of the "
            "same-region score, the hardest generalization case."
        ),
    },
    'Berlin (training region)': {
        'lat': 52.5200, 'lon': 13.4050, 'year': 2023, 'month': 6,
        'desc': (
            "Berlin IS one of the 20 cities the model trained on, unlike Rostock and Mumbai "
            "above. It's included on purpose, as the REFERENCE point: close to the model's "
            "best-case performance, on data it has already seen. Without it, there'd be no "
            "baseline to judge whether 0.45 and 0.23 above are actually good or bad."
        ),
    },
}
CUSTOM_LABEL = 'Custom location'
CUSTOM_DESC = (
    "Pick any location in the world by typing its latitude and longitude below (fields reset "
    "to 0, 0 so you don't miss one). Results will likely fall somewhere between the Rostock "
    "(mIoU 0.45, same-region) and Mumbai (mIoU 0.23, dissimilar-region) cases above, "
    "depending on how similar this place is to the training cities. Enter the coordinates "
    "and start month, then click \"Run Analysis\"."
)

custom_css = """
.gradio-container {background-color: #0B2447 !important;}
h1, h2, h3, label {color: #FFD700 !important;}
button.primary {background-color: #FFD700 !important; color: #0B2447 !important; font-weight: bold;}
.tabitem {background-color: #19376D !important;}
.final-result-box {background-color: #19376D !important; border-radius: 10px; padding: 14px 18px;}
.how-it-works-card {background-color: transparent !important; border: none !important; box-shadow: none !important; padding: 0 !important;}
.preset-desc-box {background-color: #19376D !important; border-radius: 10px; padding: 16px 20px;}
"""

# Each card: (icon, title, bullet lines, optional formula chips for card 3).
HOW_IT_WORKS_CARDS = [
    (
        "🏗️", "1. Architecture",
        [
            "Chosen as the best of <strong>8 architectures</strong> compared on a real held-out test set",
            "Trained on <strong>32 cities</strong> worldwide (20 train / 1 val / 11 held-out test)",
            f"Model: <strong>{MODEL_RUN_NAME}</strong> (UnetPlusPlus, ResNet50 encoder, SAR-pretrained)",
            f"Input: VV + VH, 256&times;256 px &rarr; Output: 1 of {NUM_CLASSES} classes per pixel",
            "Runs live in this notebook, no external server",
        ],
        None,
    ),
    (
        "🛰️", "2. Data &amp; Compositing",
        [
            "Pulls real Sentinel-1 from Earth Engine for the chosen month",
            "Takes the pixel-wise <strong>median</strong> across that month's images",
            "Same preprocessing as training (dB clip -25..5, normalize 0..1)",
        ],
        None,
    ),
    (
        "🧮", "3. Calculations",
        [
            "Turns a prediction map into plain-language, class-by-class numbers:",
            "Note: these have no ground truth to check against for a live, arbitrary query "
            "&mdash; that's exactly what the Case Study tab is for",
        ],
        [
            "Class % = pixels of that class &divide; total pixels &times; 100",
            "Month-to-month change = this month % &minus; last month %",
            "Total change = last month % &minus; first month %",
        ],
    ),
]


def _preset_desc_html(text):
    """Render a preset's explanation as a readable, medium-sized card (not tiny plain text)."""
    return (
        '<div class="preset-desc-box" style="font-size:16px;font-weight:500;'
        'color:#F5F5F5;line-height:1.55;">' + text + '</div>'
    )


def on_preset_select(name):
    if name == CUSTOM_LABEL:
        # Reset lat/lon to 0, 0 (instead of leaving the previous preset's values sitting
        # there) so it's obvious these need to be typed in, not left as someone else's city.
        return 0, 0, gr.update(), _preset_desc_html(CUSTOM_DESC)
    p = PRESETS[name]
    return p['lat'], p['lon'], f"{p['year']}-{p['month']:02d}", _preset_desc_html(p['desc'])


def on_run(lat, lon, start_month_str, progress=gr.Progress()):
    year, month = map(int, start_month_str.split('-'))
    colored_maps, pct_df = run_pipeline(lat, lon, year, month, progress=progress)
    month_to_month, total_change = compute_changes(pct_df)

    gallery_items = [(colored_maps[i], pct_df.index[i]) for i in range(12)]
    summary_md = generate_summary(pct_df, total_change)
    composition_fig = make_composition_chart(pct_df)
    stacked_fig = make_stacked_area_chart(pct_df)
    change_bar_fig = make_total_change_chart(total_change)
    # All 12 months now show up here in order, month 1 included as a row of 0.0
    # (change from itself), instead of being silently dropped.
    detailed_table = month_to_month.round(2).reset_index().rename(columns={'index': 'Month'})

    return gallery_items, summary_md, composition_fig, change_bar_fig, stacked_fig, detailed_table


def on_case_study(lat, lon, month_str):
    year, month = map(int, month_str.split('-'))
    our_map, dw_map, agreement, per_class = run_case_study(lat, lon, year, month)
    if dw_map is None:
        return our_map, None, 'No Dynamic World data available for this month/location.', ''
    breakdown_html = make_agreement_breakdown_html(per_class) if per_class else ''
    return our_map, dw_map, f'Agreement with Dynamic World: {agreement:.1f}%', breakdown_html


with gr.Blocks(css=custom_css, title='SAR Land Cover Monitor') as demo:
    gr.Markdown('# SAR Land Cover Monitor\nSentinel-1-only LULC monitoring, 12-month view, no optical imagery used.')

    with gr.Tab('How This Works'):
        gr.Markdown('### How data flows through the system')
        gr.HTML(value=make_architecture_diagram_html())
        with gr.Row():
            for icon, title, lines, formulas in HOW_IT_WORKS_CARDS:
                with gr.Column():
                    with gr.Group(elem_classes='how-it-works-card'):
                        gr.HTML(value=make_info_card_html(icon, title, lines, formulas))

    with gr.Tab('Run Analysis'):
        # Radio picker on the left, its explanation on the right. Selecting a location
        # updates lat/lon/month immediately, no separate "use preset" click needed.
        with gr.Row():
            with gr.Column(scale=1):
                preset_radio = gr.Radio(
                    choices=list(PRESETS.keys()) + [CUSTOM_LABEL],
                    value=list(PRESETS.keys())[0],
                    label='Choose a location',
                )
            with gr.Column(scale=2):
                preset_desc_out = gr.HTML(value=_preset_desc_html(list(PRESETS.values())[0]['desc']))

        with gr.Row():
            lat_in = gr.Number(label='Latitude', value=list(PRESETS.values())[0]['lat'])
            lon_in = gr.Number(label='Longitude', value=list(PRESETS.values())[0]['lon'])
            month_in = gr.Textbox(
                label='Start month (YYYY-MM)',
                value=f"{list(PRESETS.values())[0]['year']}-{list(PRESETS.values())[0]['month']:02d}",
            )

        run_btn = gr.Button('Run Analysis', variant='primary')

        preset_radio.change(on_preset_select, inputs=preset_radio,
                             outputs=[lat_in, lon_in, month_in, preset_desc_out])

        # Maps on the left, color key on the right, side by side instead of stacked.
        with gr.Row():
            with gr.Column(scale=3):
                gallery_out = gr.Gallery(label='12 Monthly LULC Maps', columns=4, height='auto')
            with gr.Column(scale=1):
                gr.Markdown('**Color key**')
                legend_out = gr.HTML(value=make_legend_html())

        gr.Markdown('## Final Result')
        with gr.Group(elem_classes='final-result-box'):
            summary_out = gr.Markdown()

        # The two quick-read charts side by side instead of one below another.
        with gr.Row():
            with gr.Column(scale=1):
                composition_out = gr.Plot(label='Composition now')
            with gr.Column(scale=1):
                change_bar_out = gr.Plot(label='Total change')

        # The 12-month trend needs the extra width, so it gets its own full-width row.
        stacked_out = gr.Plot(label='Composition over the 12 months')

        with gr.Accordion('Detailed month-to-month numbers (optional)', open=False):
            change_out = gr.Dataframe(label='Month-to-month change (%)')

        run_btn.click(on_run, inputs=[lat_in, lon_in, month_in],
                       outputs=[gallery_out, summary_out, composition_out, change_bar_out,
                                stacked_out, change_out])

    with gr.Tab('Case Study vs Dynamic World'):
        gr.Markdown(
            "**How this works:** this SAR-based model and Google's Dynamic World (a "
            "separate, already-published model, built from optical satellite images "
            "instead of radar) each predict the land cover for the exact same location "
            "and month, independently. Dynamic World's categories are mapped onto this "
            "project's 9 classes, then the two predictions are compared pixel by pixel. "
            "The % below is the real, live agreement between the two for this location, "
            "not a fixed number."
        )
        with gr.Row():
            cs_lat = gr.Number(label='Latitude', value=54.0887)
            cs_lon = gr.Number(label='Longitude', value=12.1414)
            cs_month = gr.Textbox(label='Month (YYYY-MM)', value='2023-06')
        cs_btn = gr.Button('Compare', variant='primary')
        with gr.Row():
            cs_our = gr.Image(label='Your Model')
            cs_dw = gr.Image(label='Dynamic World (mapped to your classes)')
        cs_agreement = gr.Textbox(label='Result')
        cs_breakdown = gr.HTML()

        cs_btn.click(on_case_study, inputs=[cs_lat, cs_lon, cs_month],
                     outputs=[cs_our, cs_dw, cs_agreement, cs_breakdown])

demo.launch(share=True, debug=True)


**Notes before you record the video:**
- First run per location takes about 1-2 minutes (12 real Earth Engine calls + model inference). That's expected, the status bar shows progress through all 12 months.
- If a preset location gives 'No Sentinel-1 images found' for a month, try a nearby month, Sentinel-1 doesn't image every location every single month.
- The `share=True` link works as long as this Colab cell keeps running, closing the tab or the session ending will break the link.
- For a smoother recording, run the Rostock/Mumbai/Berlin presets once before recording so you know they work, then you can either replay those or do one live for the camera.